In [ ]:
import json
import os
import shutil
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path

# T4-oriented single-cell launcher for Phase B.
# Edit only RUN_MODE if you are advancing from SMOKE -> DIAGNOSE -> FULL.

RUN_MODE = "SMOKE"
REPO_ROOT = ""
GIT_CLONE_URL = "https://github.com/dttdrv/gerhard.git"
GIT_CLONE_BRANCH = "main"
EXPLICIT_CHECKPOINT_PATH = ""
AUTO_MOUNT_DRIVE = True
AUTO_DOWNLOAD_BUNDLE = True

CHECKPOINT_SEARCH_DIRS = [
    "/content/drive/MyDrive/gerhard/checkpoints",
    "/content/drive/MyDrive/gerhard",
    "/content/drive/MyDrive/checkpoints",
    "/content/checkpoints",
    "/content",
]
CHECKPOINT_NAME_HINTS = [
    "v15_best.pt",
    "v15.pt",
    "*v15*.pt",
    "*v14.3*.pt",
    "*.pt",
    "*.pth",
    "*.ckpt",
]


def _run(cmd, cwd=None, env=None, check=True):
    printable = " ".join(str(part) for part in cmd)
    print(f"$ {printable}")
    return subprocess.run(cmd, cwd=cwd, env=env, check=check, text=True)


def _ensure_jupyter():
    try:
        _run([sys.executable, "-m", "jupyter", "--version"])
    except Exception:
        _run([sys.executable, "-m", "pip", "install", "-q", "jupyter", "nbconvert", "ipykernel"])


def _mount_drive_if_needed():
    if "google.colab" in sys.modules and AUTO_MOUNT_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)


def _find_repo_root() -> Path:
    candidates = []
    if REPO_ROOT:
        candidates.append(Path(REPO_ROOT).expanduser())
    candidates.extend([
        Path("/content/gerhard"),
        Path("/content/drive/MyDrive/gerhard"),
        Path.cwd(),
        *Path.cwd().parents,
    ])
    seen = set()
    for candidate in candidates:
        candidate = candidate.resolve()
        key = str(candidate)
        if key in seen:
            continue
        seen.add(key)
        if (
            (candidate / "notebooks" / "asnn_goose_v15_reset_master.ipynb").exists()
            and (candidate / "scripts" / "register_dossier_run.py").exists()
        ):
            return candidate
    target = Path("/content/gerhard")
    if target.exists():
        shutil.rmtree(target)
    _run(["git", "clone", "--branch", GIT_CLONE_BRANCH, GIT_CLONE_URL, str(target)])
    return target.resolve()


def _gpu_name() -> str:
    try:
        result = _run([
            "nvidia-smi",
            "--query-gpu=name",
            "--format=csv,noheader",
        ])
        return result.stdout.strip().splitlines()[0].strip()
    except Exception:
        return "unknown"


def _default_batch_size(gpu_name: str) -> int:
    if "T4" in gpu_name.upper():
        return 4
    return 8


def _find_checkpoint() -> Path:
    if EXPLICIT_CHECKPOINT_PATH:
        candidate = Path(EXPLICIT_CHECKPOINT_PATH).expanduser()
        if candidate.exists():
            return candidate.resolve()
        raise FileNotFoundError(f"Explicit checkpoint path not found: {candidate}")

    matches = []
    for root_text in CHECKPOINT_SEARCH_DIRS:
        root = Path(root_text)
        if not root.exists():
            continue
        for pattern in CHECKPOINT_NAME_HINTS:
            found = sorted(root.rglob(pattern))
            for path in found:
                if path.is_file():
                    matches.append(path.resolve())
            if matches:
                break
        if matches:
            break

    if matches:
        chosen = sorted(matches, key=lambda p: ("v15" not in p.name.lower(), len(str(p))))[0]
        return chosen

    if "google.colab" in sys.modules:
        from google.colab import files
        print("No checkpoint found automatically. Upload one now.")
        uploaded = files.upload()
        if uploaded:
            upload_dir = Path("/content/checkpoints")
            upload_dir.mkdir(parents=True, exist_ok=True)
            uploaded_name = next(iter(uploaded))
            src = Path("/content") / uploaded_name
            dst = upload_dir / uploaded_name
            if src.exists() and src != dst:
                shutil.move(str(src), str(dst))
            return dst.resolve()

    raise FileNotFoundError(
        "Could not find a checkpoint automatically. Set EXPLICIT_CHECKPOINT_PATH or upload one."
    )


_mount_drive_if_needed()
_ensure_jupyter()

if RUN_MODE not in {"SMOKE", "DIAGNOSE", "FULL"}:
    raise ValueError(f"RUN_MODE must be one of SMOKE/DIAGNOSE/FULL, got: {RUN_MODE}")

gpu_name = _gpu_name()
batch_size = _default_batch_size(gpu_name)
print(f"GPU: {gpu_name}")
print(f"Batch size default for this GPU: {batch_size}")

checkpoint_file = _find_checkpoint()
print(f"Checkpoint: {checkpoint_file}")

repo_root = _find_repo_root()
reset_notebook = repo_root / "notebooks" / "asnn_goose_v15_reset_master.ipynb"
if not reset_notebook.exists():
    raise FileNotFoundError(f"Missing canonical reset notebook: {reset_notebook}")

run_id_prefix = {
    "SMOKE": "v15_preflight_smoke",
    "DIAGNOSE": "v15_preflight_diagnose",
    "FULL": "v15_full",
}[RUN_MODE]
run_id = f"{run_id_prefix}_{datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')}"

operator_dir = repo_root / "outputs" / "operator_runs" / run_id
operator_dir.mkdir(parents=True, exist_ok=True)
executed_notebook_name = f"{run_id}_{RUN_MODE.lower()}_executed.ipynb"

exec_env = os.environ.copy()
exec_env.update(
    {
        "GERHARD_RUN_MODE": RUN_MODE,
        "GERHARD_RUN_ID": run_id,
        "GERHARD_CHECKPOINT_PATH": str(checkpoint_file),
        "GERHARD_ENABLE_DOSSIER_EXPORT": "1",
        "GERHARD_ENABLE_REGISTER_RUN": "0",
        "GERHARD_ENABLE_AUTODOWNLOAD_DOSSIER": "0",
        "GERHARD_BATCH_SIZE": str(batch_size),
        "GERHARD_SMOKE_BATCHES": "2",
        "GERHARD_FULL_BATCHES_SMOKE": "4",
        "GERHARD_FULL_BATCHES_DIAGNOSE": "40",
        "GERHARD_FULL_BATCHES": "20",
    }
)

(operator_dir / "operator_env.json").write_text(
    json.dumps({k: exec_env[k] for k in sorted(exec_env) if k.startswith("GERHARD_")}, indent=2),
    encoding="utf-8",
)

cmd = [
    sys.executable,
    "-m",
    "jupyter",
    "nbconvert",
    "--to",
    "notebook",
    "--execute",
    str(reset_notebook),
    "--ExecutePreprocessor.timeout=-1",
    "--output",
    executed_notebook_name,
    "--output-dir",
    str(operator_dir),
]

result = _run(cmd, cwd=str(repo_root), env=exec_env, check=False)
if result.returncode != 0:
    raise RuntimeError(f"Reset notebook execution failed with exit code {result.returncode}")

run_dir = repo_root / "outputs" / run_id
expected = [
    "eval_suite.json",
    "metrics.json",
    "config.yaml",
    "seed.txt",
    "v15_spikingbrain.json",
    f"run_dossier_{run_id}.html",
]
missing = [name for name in expected if not (run_dir / name).exists()]

print(f"\nRUN_ID: {run_id}")
print(f"Artifact dir: {run_dir}")
print(f"Executed notebook: {operator_dir / executed_notebook_name}")
print("\nArtifacts:")
for name in expected:
    path = run_dir / name
    print(f"- {name}: {'present' if path.exists() else 'missing'} -> {path}")

if missing:
    print("\nStructural stop: missing expected artifacts.")
    print("Do not continue to the next mode until you inspect the partial bundle.")
else:
    dossier_path = run_dir / f"run_dossier_{run_id}.html"
    archive_base = operator_dir / f"{run_id}_bundle"
    bundle_zip = shutil.make_archive(str(archive_base), "zip", run_dir)
    print("\nBundle ready.")
    print(f"ZIP: {bundle_zip}")
    print("Laptop-side registration command:")
    print(f'python scripts/register_dossier_run.py --dossier "{dossier_path}" --phase B')
    if AUTO_DOWNLOAD_BUNDLE and "google.colab" in sys.modules:
        from google.colab import files
        files.download(bundle_zip)
